In [20]:
import os
import pandas as pd
import geopandas as gpd

# Section 1: Initial processing
# Set the working directory
os.chdir("/home/jovyan/work/Typhoon_IBF_Rice_Damage_Model/")
cdir = os.getcwd()

# Load the first CSV file
typhoon_municipality_df = pd.read_csv('IBF_typhoon_model/data/gis_data/typhoon_enter_exit_per_municipality.csv')

# Convert entry_date and exit_date to datetime format
typhoon_municipality_df['entry_date'] = pd.to_datetime(typhoon_municipality_df['entry_date'])
typhoon_municipality_df['exit_date'] = pd.to_datetime(typhoon_municipality_df['exit_date'])

# Calculate the earliest entry date/time and latest exit date/time for each unique name_year
earliest_entry = typhoon_municipality_df.groupby('name_year')['entry_date'].min().reset_index()
latest_exit = typhoon_municipality_df.groupby('name_year')['exit_date'].max().reset_index()

# Merge the earliest entry and latest exit times into a single DataFrame
time_summary_df = pd.merge(earliest_entry, latest_exit, on='name_year', how='inner')
time_summary_df.rename(columns={
    'entry_date': 'earliest_entry_date',
    'exit_date': 'latest_exit_date'
}, inplace=True)

# Extract date and time in the required formats
time_summary_df['earliest_entry_time'] = time_summary_df['earliest_entry_date'].dt.strftime('%H:%M:%S')
time_summary_df['earliest_entry_date'] = time_summary_df['earliest_entry_date'].dt.strftime('%d/%m/%Y')
time_summary_df['latest_exit_time'] = time_summary_df['latest_exit_date'].dt.strftime('%H:%M:%S')
time_summary_df['latest_exit_date'] = time_summary_df['latest_exit_date'].dt.strftime('%d/%m/%Y')

# Load the second CSV file
metadata_typhoons_df = pd.read_csv('IBF_typhoon_model/data/rainfall_data/input/metadata_typhoons.csv')

# Merge the time summary data with the metadata typhoons data
updated_metadata_df = pd.merge(metadata_typhoons_df, time_summary_df, left_on='typhoon', right_on='name_year', how='left')

# Load the Excel file and read the second sheet (typhoon_overview)
data_overview_df = pd.read_excel('IBF_typhoon_model/data/data_overview.xlsx', sheet_name='typhoon_overview')

# Merge the SID from the data overview file
updated_metadata_with_sid_df = pd.merge(updated_metadata_df, data_overview_df[['name_year', 'storm_id']], left_on='typhoon', right_on='name_year', how='left')

# Drop the redundant name_year_y column and rename storm_id to SID
updated_metadata_with_sid_df = updated_metadata_with_sid_df.drop(columns=['name_year_y', 'name_year_x'])
updated_metadata_with_sid_df.rename(columns={'storm_id': 'SID'}, inplace=True)

# Save the updated metadata to a new CSV file
updated_metadata_with_sid_df.to_csv('IBF_typhoon_model/data/rainfall_data/input/updated_metadata_typhoons_with_sid.csv', index=False)

# Section 2: Additional processing
# Load the shapefile
tracks_df = gpd.read_file('IBF_typhoon_model/data/gis_data/typhoon_tracks/tracks_filtered.shp')

# Convert ISO_time to datetime format
tracks_df['ISO_TIME'] = pd.to_datetime(tracks_df['ISO_TIME'])

# Calculate the earliest and latest ISO_time for each SID
earliest_iso = tracks_df.groupby('SID')['ISO_TIME'].min().reset_index()
latest_iso = tracks_df.groupby('SID')['ISO_TIME'].max().reset_index()

# Merge the earliest and latest ISO times into a single DataFrame
iso_summary_df = pd.merge(earliest_iso, latest_iso, on='SID', how='inner')
iso_summary_df.rename(columns={
    'ISO_TIME_x': 'start_time',
    'ISO_TIME_y': 'end_time'
}, inplace=True)

# Ensure start_time and end_time are in datetime format
iso_summary_df['start_time'] = pd.to_datetime(iso_summary_df['start_time'])
iso_summary_df['end_time'] = pd.to_datetime(iso_summary_df['end_time'])

# Extract date and time in the required formats
iso_summary_df['start_date'] = iso_summary_df['start_time'].dt.strftime('%d/%m/%Y')
iso_summary_df['start_time'] = iso_summary_df['start_time'].dt.strftime('%H:%M:%S')
iso_summary_df['end_date'] = iso_summary_df['end_time'].dt.strftime('%d/%m/%Y')
iso_summary_df['end_time'] = iso_summary_df['end_time'].dt.strftime('%H:%M:%S')

# Merge the new time summary data with the updated metadata
final_metadata_df = pd.merge(updated_metadata_with_sid_df, iso_summary_df, on='SID', how='left')

# Reorganize the columns
final_metadata_df = final_metadata_df[['typhoon', 'SID', 'startdate', 'start_time', 'enddate', 'end_time', 
                                       'landfalldate', 'landfall_time', 'earliest_entry_date', 
                                       'earliest_entry_time', 'latest_exit_date', 'latest_exit_time', 
                                       'imerg_type']]

# Print out all the column names
print(final_metadata_df.columns)

# Save the final updated metadata to a new CSV file
final_metadata_df.to_csv('IBF_typhoon_model/data/rainfall_data/input/expanded_metadata_typhoons.csv', index=False)


Index(['typhoon', 'SID', 'startdate', 'start_time', 'enddate', 'end_time',
       'landfalldate', 'landfall_time', 'earliest_entry_date',
       'earliest_entry_time', 'latest_exit_date', 'latest_exit_time',
       'imerg_type'],
      dtype='object')


#### Comparing the earliest entry with the landfall for each typhoon.


In [39]:
import pandas as pd

# Load the final updated metadata CSV file
final_metadata_df = pd.read_csv('IBF_typhoon_model/data/rainfall_data/input/expanded_metadata_typhoons.csv')

# Function to combine date and time into a single datetime column
def combine_datetime(df, date_col, time_col, new_col_name):
    df[new_col_name] = pd.to_datetime(df[date_col] + ' ' + df[time_col], format='%d/%m/%Y %H:%M:%S')

# Function to calculate the time difference and add formatted columns
def calculate_time_difference(df, start_col, end_col, diff_col_name):
    df[diff_col_name] = df[end_col] - df[start_col]
    df[f'{diff_col_name}_total_minutes'] = df[diff_col_name].dt.total_seconds() // 60
    df[f'{diff_col_name}_hours_minutes'] = (df[f'{diff_col_name}_total_minutes'] // 60).astype(str) + ' hours'
    df[f'{diff_col_name}_days_hours_minutes'] = (df[f'{diff_col_name}_total_minutes'] // (60 * 24)).astype(str) + ' days ' + (df[f'{diff_col_name}_total_minutes'] // 60 % 24).astype(str) + ' hours'

# Combine datetime columns
combine_datetime(final_metadata_df, 'enddate', 'end_time', 'end_datetime')
combine_datetime(final_metadata_df, 'latest_exit_date', 'latest_exit_time', 'latest_exit_datetime')
combine_datetime(final_metadata_df, 'startdate', 'start_time', 'start_datetime')
combine_datetime(final_metadata_df, 'landfalldate', 'landfall_time', 'landfall_datetime')
combine_datetime(final_metadata_df, 'earliest_entry_date', 'earliest_entry_time', 'earliest_entry_datetime')

# Calculate time differences
calculate_time_difference(final_metadata_df, 'latest_exit_datetime', 'end_datetime', 'exit_end_diff')
calculate_time_difference(final_metadata_df, 'start_datetime', 'landfall_datetime', 'start_landfall_diff')
calculate_time_difference(final_metadata_df, 'earliest_entry_datetime', 'landfall_datetime', 'entry_landfall_diff')

# Print results
def print_time_differences(df, typhoon_col, diff_col_name):
    print(f'{diff_col_name.replace("_", " ")}')
    print('-------------------------------------------------------------')
    for index, row in df.iterrows():
        typhoon_name = row[typhoon_col]
        total_minutes = row[f'{diff_col_name}_total_minutes']
        hours_minutes = row[f'{diff_col_name}_hours_minutes']
        days_hours_minutes = row[f'{diff_col_name}_days_hours_minutes']
        
        print(f'Typhoon: {typhoon_name}   | {total_minutes} minutes   | {hours_minutes}   | {days_hours_minutes}')
        print('---------------------------------------------------------------------------------------------------')

print_time_differences(final_metadata_df, 'typhoon', 'entry_landfall_diff')
print_time_differences(final_metadata_df, 'typhoon', 'exit_end_diff')
print_time_differences(final_metadata_df, 'typhoon', 'start_landfall_diff')



entry landfall diff
-------------------------------------------------------------
Typhoon: aere2011   | 2520.0 minutes   | 42.0 hours   | 1.0 days 18.0 hours
---------------------------------------------------------------------------------------------------
Typhoon: atsani2020   | 900.0 minutes   | 15.0 hours   | 0.0 days 15.0 hours
---------------------------------------------------------------------------------------------------
Typhoon: bopha2012   | 720.0 minutes   | 12.0 hours   | 0.0 days 12.0 hours
---------------------------------------------------------------------------------------------------
Typhoon: danas2019   | 1800.0 minutes   | 30.0 hours   | 1.0 days 6.0 hours
---------------------------------------------------------------------------------------------------
Typhoon: durian2006   | 1800.0 minutes   | 30.0 hours   | 1.0 days 6.0 hours
---------------------------------------------------------------------------------------------------
Typhoon: fung-wong2014   | 2340.0 mi